# Vision Transformer (ViT) Helmet Classifier Training on GPU

This notebook fine-tunes **google/vit-base-patch16-224** for binary helmet compliance verification (`helmet` vs `no_helmet`) using **Google Colab Pro GPU** (T4 / V100 / A100).

### Steps to Run:
1. **Ensure GPU is Active**: Click **Runtime** -> **Change runtime type** -> Select **T4 GPU** or **A100 GPU** -> Save.
2. **Upload `vit_dataset.zip`**: Upload the `vit_dataset.zip` file generated from your local machine to the Colab files pane on the left.
3. **Run All Cells**: Click **Runtime** -> **Run all** (takes ~5–7 minutes).
4. **Download**: The final cell will automatically download `vit_helmet_model.zip` for use in your project.

In [ ]:
# Step 1: Check GPU & CUDA Acceleration
!nvidia-smi
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected! Go to Runtime -> Change runtime type -> Select GPU')

In [ ]:
# Step 2: Install required Hugging Face libraries
!pip install -q transformers datasets evaluate accelerate torchvision
print('Libraries installed successfully!')

In [ ]:
# Step 3: Unzip the uploaded vit_dataset.zip
import os, zipfile

if not os.path.exists('vit_dataset.zip'):
    print('Please upload vit_dataset.zip to the Colab file pane on the left!')
else:
    with zipfile.ZipFile('vit_dataset.zip', 'r') as z:
        z.extractall('.')
    print('Dataset extracted successfully!')
    for split in ['train', 'val']:
        p = os.path.join('data', 'vit_dataset', split)
        if os.path.exists(p):
            counts = {c: len(os.listdir(os.path.join(p, c))) for c in os.listdir(p)}
            print(f'{split.upper()} split count: {counts}')

In [ ]:
# Step 4: Setup Dataset Loaders and Transforms
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from transformers import AutoImageProcessor, AutoModelForImageClassification, TrainingArguments, Trainer
import evaluate
import numpy as np

MODEL_ID = 'google/vit-base-patch16-224'
image_processor = AutoImageProcessor.from_pretrained(MODEL_ID)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=image_processor.image_mean, std=image_processor.image_std)
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=image_processor.image_mean, std=image_processor.image_std)
])

train_dataset = ImageFolder('data/vit_dataset/train', transform=train_transform)
val_dataset = ImageFolder('data/vit_dataset/val', transform=val_transform)

id2label = {i: c for i, c in enumerate(train_dataset.classes)}
label2id = {c: i for i, c in enumerate(train_dataset.classes)}
print('Classes:', id2label)
print('Total Training Samples:', len(train_dataset))
print('Total Validation Samples:', len(val_dataset))

In [ ]:
# Step 5: Initialize ViT Model with Binary Helmet Head
model = AutoModelForImageClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)['accuracy']
    f1 = f1_metric.compute(predictions=predictions, references=labels, average='weighted')['f1']
    return {'accuracy': acc, 'f1': f1}

print('ViT model initialized with 2-class head.')

In [ ]:
# Step 6: Configure GPU Training Arguments and Launch Training
import torch

def collate_fn(batch):
    pixel_values = torch.stack([item[0] for item in batch])
    labels = torch.tensor([item[1] for item in batch], dtype=torch.long)
    return {'pixel_values': pixel_values, 'labels': labels}

training_args = TrainingArguments(
    output_dir='./vit_checkpoints',
    logging_dir='./logs',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=10,
    save_total_limit=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
    compute_metrics=compute_metrics
)

print('Starting ViT fine-tuning on GPU...')
trainer.train()
print('Training complete!')

In [ ]:
# Step 7: Evaluate on Validation Set
eval_results = trainer.evaluate()
print('\n--- Final Validation Metrics ---')
for k, v in eval_results.items():
    print(f'{k}: {v:.4f}' if isinstance(v, float) else f'{k}: {v}')

In [ ]:
# Step 8: Save Model, Zip, and Download for Local Deployment
import shutil
from google.colab import files

SAVE_DIR = './vit_helmet_model'
trainer.save_model(SAVE_DIR)
image_processor.save_pretrained(SAVE_DIR)

# Zip model folder
shutil.make_archive('vit_helmet_model', 'zip', SAVE_DIR)
print('Saved model zipped to vit_helmet_model.zip!')
print('Triggering download to your computer...')
files.download('vit_helmet_model.zip')